In [ ]:
import json
import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


## Load data files

In [ ]:
all_files = glob.glob("../data/**/*.json", recursive=True)
game_files = [
    f for f in all_files
    if "users" not in f and "sessions" not in f
]

test_files = game_files[:5]
selected_files = test_files

for file_path in selected_files:
    with open(file_path) as file:
        game = json.load(file)
    n_mouse = sum(1 for e in game["events"] if e.get("type") == "MouseEvent")
    print(f"userId={game['userId']}, gameId={game['id']}, mouseEvents={n_mouse}, file={Path(file_path).name}")


## Mouse trajectory preview

In [ ]:
for i, file_path in enumerate(selected_files):
    with open(file_path) as f:
        game_data = json.load(f)

    mouse_events = [
        event for event in game_data["events"]
        if event.get("type") == "MouseEvent"
    ]
    user_id = game_data.get("userId")
    game_id = game_data.get("id")

    df = pd.DataFrame(mouse_events)
    df["trajectory_x"] = df["dx"].cumsum()
    df["trajectory_y"] = df["dy"].cumsum()

    print(f"\nuserId={user_id}, gameId={game_id}, events={len(df)}")
    # print(df[["time", "dx", "dy", "trajectory_x", "trajectory_y"]].head())

    plt.subplot(1, 5, i+1)
    plt.plot(df["trajectory_x"], df["trajectory_y"], linewidth=0.5)
    plt.title(f"userId={user_id}, gameId={game_id}")

plt.tight_layout()
plt.show()

## Feature extraction


In [ ]:
game_rows = []

for file_path in game_files:
    with open(file_path) as file:
        game_data = json.load(file)

    mouse = [e for e in game_data["events"] if e.get("type") == "MouseEvent"]
    if len(mouse) < 2:
        continue

    df = pd.DataFrame(mouse).sort_values("time")

    distance = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)
    dt = df["time"].diff()
    valid = dt > 0
    speed = distance[valid] / dt[valid]
    angles = np.arctan2(df["dy"], df["dx"]).diff().abs()[valid]

    game_rows.append({
        "userId": game_data["userId"],
        "gameId": game_data["id"],
        "source_file": os.path.basename(file_path),
        "total_movement": distance.sum(),
        "avg_speed": speed.mean(),
        "n_events": len(df),
        "mean_dt": dt[valid].mean(),
        "idle_ratio": (distance < 1).mean(),
        "avg_turn_angle": angles.mean(),
        "speed_std": speed.std(),
        "speed_max": speed.max(),
        "dist_std": distance.std(),
    })

games_df = pd.DataFrame(game_rows)
games_df.to_csv("../data/red_eclipse_features.csv", index=False)
print(games_df.head())


## Player identification


In [ ]:
feature_cols = [
    "total_movement", "avg_speed", "n_events", "mean_dt",
    "idle_ratio", "avg_turn_angle", "speed_std", "speed_max", "dist_std",
]

MIN_GAMES = 8
games_df_37 = games_df.groupby("userId").filter(lambda g: len(g) >= MIN_GAMES)

# games_df = 45 players, games_df_37 = more than 8 games
select_model = games_df_37

input_data = select_model[feature_cols]
output_data = select_model["userId"]

input_train, input_test, output_train, output_test = train_test_split(
    input_data, output_data, test_size=0.2, random_state=42, stratify=output_data
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(input_train, output_train)

output_pred = model.predict(input_test)
accuracy = accuracy_score(output_test, output_pred)

print(f"Accuracy: {accuracy:.2%}")
print(f"Random baseline: {1 / output_data.nunique():.2%} ({output_data.nunique()} players, {len(select_model)} games)")
print()
print(classification_report(output_test, output_pred))
